In [3]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'#https://blog.csdn.net/weixin_46713695/article/details/130134955 OpenBLAS warning
import mudata as md
import scvi
from scvi.model import TOTALVI
import argparse
from sklearn.cluster import KMeans
from time import time
import scanpy as sc
import numpy as np
import pandas as pd
import h5py
scvi.settings.seed = 1000

Seed set to 1000


In [8]:
from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.metrics.cluster import normalized_mutual_info_score as nmi_score
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, adjusted_mutual_info_score
from sklearn.metrics import fowlkes_mallows_score, homogeneity_score, completeness_score, v_measure_score, silhouette_score

def eva(y_true, y_pred):
    nmi = nmi_score(y_true, y_pred)
    ari = ari_score(y_true, y_pred)
    ami = adjusted_mutual_info_score(y_true, y_pred)
    fmi = fowlkes_mallows_score(y_true, y_pred)
    hom = homogeneity_score(y_true, y_pred)
    com = completeness_score(y_true, y_pred)
    v = v_measure_score(y_true, y_pred)
    return nmi, ari, ami, fmi, hom, com, v

In [ ]:
def read_data(file_path1, file_path2, file_type, label_file):
    if file_type=='h5ad':
        atac = sc.read_h5ad(file_path2)
        rna = sc.read_h5ad(file_path1)
        atac_X = np.array(atac.X.toarray())#
        rna_X = np.array(rna.X.toarray())
        if label_file==None:
            cell_name = np.array(atac.obs["cell_type"])
        else:
            cell_name = pd.read_csv(label_file, usecols=[1])
        cell_type, y = np.unique(cell_name, return_inverse=True)
        print(y)
        cluster_number = int(max(y) - min(y) + 1)   
        adata_RNA = sc.AnnData(rna_X)
        adata_ATAC = sc.AnnData(atac_X)
    elif file_type=='h5':
        data_mat = h5py.File(file_path1)
        rna_X = np.array(data_mat['X1'])
        atac_X = np.array(data_mat['X2'])
        y = np.array(data_mat['Y'])
        data_mat.close()
        cluster_number = int(max(y) - min(y) + 1) 
        adata_RNA = sc.AnnData(rna_X)
        adata_ATAC = sc.AnnData(atac_X)
    elif file_type=='loom':
        adata_RNA=sc.read_loom(file_path1)
        adata_ATAC=sc.read_loom(file_path2)
        cell_name = pd.read_csv(label_file, usecols=[1])
        cell_type, y = np.unique(cell_name, return_inverse=True)
        cluster_number = int(max(y) - min(y) + 1)   

    return adata_RNA, adata_ATAC, cluster_number, y

In [ ]:
adata_RNA, adata_ATAC, cluster_number, y = read_data('./inhouse_rna.h5ad', 
                                                     './inhouse_adt.h5ad', 
                                                     'h5ad', 
                                                     None)

t0=time()
mdata = md.MuData({"rna": adata_RNA, "protein": adata_ATAC})
TOTALVI.setup_mudata(mdata,
                                rna_layer=None,
                                protein_layer=None,
                                modalities={"rna_layer": "rna",
                                            "protein_layer": "protein"})
vae = TOTALVI(mdata)
vae.train()
latent = vae.get_latent_representation()
kmeans = KMeans(n_clusters = cluster_number, n_init=20)
y_pred = kmeans.fit_predict(latent)   

/data1/chengyue/miniconda3/envs/cmvae/lib/python3.9/site-packages/mudata/_core/mudata.py:489: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


[4 0 3 ... 6 3 4]


Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A800-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/data1/chengyue/miniconda3/envs/cmvae/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the

Epoch 267/400:  67%|██████▋   | 267/400 [01:25<00:42,  3.12it/s, v_num=1, train_loss_step=2.98e+3, train_loss_epoch=2.86e+3]
Monitored metric elbo_validation did not improve in the last 45 records. Best score: 3436.594. Signaling Trainer to stop.


In [9]:
from sklearn.metrics import silhouette_score
nmi, ari, ami, fmi, hom, com, v = eva(y, y_pred) 
print('z for clustering, NMI:{:.4f}, ARI:{:.4f}, AMI:{:.4f}, FMI:{:.4f}, HOM:{:.4f}, COM:{:.4f}, V:{:.4f}'.format(nmi, ari, ami, fmi, hom, com, v))
t1=time()
t=t1-t0
print('time is:{:.4f}'.format(t))

z for clustering, NMI:0.5239, ARI:0.3917, AMI:0.5200, FMI:0.5249, HOM:0.5648, COM:0.4884, V:0.5239
time is:256.1681
